In [ ]:
import os
import pandas as pd
import numpy as np
import pyspark
from pyspark.sql import SparkSession


In [ ]:
spark = SparkSession.builder.appName("OlistDataAnalysis").getOrCreate()
print("SparkSession initialized.")

olist_pd_orders_cleaned_df = spark.read.csv('/content/olist_orders_cleaned.csv', header=True, inferSchema=True)
print('olist_orders_cleaned loaded.')

SparkSession initialized.
olist_orders_cleaned loaded.


### Limpeza e Análise com Pandas: `olist_orders_dataset.csv`

In [ ]:
# Carregar o dataset principal com pandas
pd_orders_df = pd.read_csv('/content/olist_orders_dataset.csv')

print('Primeiras 5 linhas do DataFrame:')
display(pd_orders_df.head())

print('\nInformações do DataFrame (incluindo tipos de dados e valores não nulos):')
display(pd_orders_df.info())

print('\nContagem de valores nulos por coluna:')
display(pd_orders_df.isnull().sum())

Primeiras 5 linhas do DataFrame:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Informações do DataFrame (incluindo tipos de dados e valores não nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


None


Contagem de valores nulos por coluna:


,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


### Script de Limpeza de Dados com Pandas (Exemplo para `olist_orders_dataset.csv`)

Vamos converter as colunas de data para o tipo datetime e, para fins de demonstração, preencher os valores nulos das datas de aprovação e entrega com uma estratégia simples (por exemplo, preenchendo com a data de compra mais um delta).

In [ ]:
pd_orders_cleaned_df = pd_orders_df.copy()

# Converter colunas de timestamp para datetime
date_cols_pd = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col_name in date_cols_pd:
    pd_orders_cleaned_df[col_name] = pd.to_datetime(pd_orders_cleaned_df[col_name])

# Preencher valores nulos para 'order_approved_at'
# Se 'order_approved_at' for nulo, preencher com
# 'order_purchase_timestamp' + 1 dia
pd_orders_cleaned_df['order_approved_at'] = pd_orders_cleaned_df['order_approved_at'].fillna(
    pd_orders_cleaned_df['order_purchase_timestamp'] + pd.Timedelta(days=1)
)

# Preencher valores nulos para 'order_delivered_carrier_date'
# Se 'order_delivered_carrier_date' for nulo, preencher com
#'order_approved_at' + 1 dia
pd_orders_cleaned_df['order_delivered_carrier_date'] = pd_orders_cleaned_df['order_delivered_carrier_date'].fillna(
    pd_orders_cleaned_df['order_approved_at'] + pd.Timedelta(days=1)
)

# Preencher valores nulos para 'order_delivered_customer_date'
# Se 'order_delivered_customer_date' for nulo, preencher com
#'order_delivered_carrier_date' + 2 dias
pd_orders_cleaned_df['order_delivered_customer_date'] = pd_orders_cleaned_df['order_delivered_customer_date'].fillna(
    pd_orders_cleaned_df['order_delivered_carrier_date'] + pd.Timedelta(days=2)
)

print('DataFrame pandas após limpeza básica:')
display(pd_orders_cleaned_df.head())

print('\nContagem de valores nulos após limpeza:')
display(pd_orders_cleaned_df.isnull().sum())

pd_orders_cleaned_df.to_csv('olist_orders_cleaned.csv', index=False)

DataFrame pandas após limpeza básica:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26



Contagem de valores nulos após limpeza:


,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,0
order_delivered_carrier_date,0
order_delivered_customer_date,0
order_estimated_delivery_date,0


### PySpark SQL: Olhar a tabela limpa e seus tipos

In [ ]:
# Registrar todos os DataFrames PySpark como tabelas temporárias para SQL
olist_pd_orders_cleaned_df.createOrReplaceTempView("orders")
print("Tabelas temporárias PySpark SQL criadas.")

# Query para listar todas as tabelas e seus esquemas
all_tables = spark.catalog.listTables()

for table in all_tables:
    if table.isTemporary:
        print(f"\n--- Esquema da Tabela '{table.name}' ---")
        spark.sql(f"DESCRIBE {table.name}").show()

print("\nPrimeiras 5 linhas da Tabela 'orders_cleaned' (após limpeza) para verificar os dados:")
spark.sql("SELECT * FROM orders LIMIT 5").show()

Tabelas temporárias PySpark SQL criadas.

--- Esquema da Tabela 'category_translation' ---
+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|product_category_...|   string|   NULL|
|product_category_...|   string|   NULL|
+--------------------+---------+-------+


--- Esquema da Tabela 'customers' ---
+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|         customer_id|   string|   NULL|
|  customer_unique_id|   string|   NULL|
|customer_zip_code...|      int|   NULL|
|       customer_city|   string|   NULL|
|      customer_state|   string|   NULL|
+--------------------+---------+-------+


--- Esquema da Tabela 'geolocation' ---
+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|geolocation_zip_c...|      int|   NULL|
|     geolocation_lat|   double|   NULL|
|     g

In [ ]:
spark.sql("""
SELECT
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM orders
""").show()

+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|                   2017-10-18|
|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|                   2018-08-13|
|   delivered|     2018-08-08 08:38:49|2018-08-08 08:55:23|         2018-08-08 13:50:00|          2018-08-17 18:06:29|                   2018-09-04|
|   delivered|     2017-11-18 19:28:06|2017-11-18 19:45:59|         2017-11-22 13:39:59|          2017-12-